# TinyDB Chat Room

### Overview

This project creates a TinyDB chat room database that tracks users and messages as users create posts. I used two related databases connected through `user_handle`: the **Users** database stores each user's unique handle, age for verification, and interests, while the **Posts** database stores the user's handle, age, post, and time of the post. Keeping them separate keeps the Users database lighter and creates a **one-to-many relationship** between users and posts, while interests create a **many-to-many relationship** because users can have multiple interests and each interest can belong to multiple users. The `create_user()` function prevents duplicate handles and stores age and interests, `match_user()` verifies a user using both handle and age with a case-insensitive query, and `create_post()` verifies the user before inserting a post. For aggregation and reporting, I counted posts per user, calculated the average number of posts, and counted posts from users with a specific interest. I also tested database performance with 1,000 users and 100,000 posts; adding the test data and running the simple queries each took less than one second. TinyDB worked well for this chat room application, although further testing could determine when performance begins to slow down significantly as more posts are added.


# Create DB and users: 

In [158]:
from tinydb import TinyDB, Query

In [159]:
posts = TinyDB('Posts.json')
users = TinyDB('Users.json')

In [131]:
# Insert new user into Users.json
from tinydb import where # this is instead of inti queary tool
import re

def create_user():
    user_handle = input("Enter user handle: ")

    # check if user handle exsits for database cleanliness 
    matched_user = users.search(   
        where('user_handle').matches(
            user_handle,
            flags=re.IGNORECASE
        )
    )

    # If the search found a user, the handle is already taken
    if matched_user:
        print("User handle taken.")
        return

    user_age = int(input("Enter user age: "))

    interests = input(
        "Enter two interests separated by a comma with no space "
        "(e.g. sports,python): "
    ).split(",")

    users.insert({
        "user_handle": user_handle,
        "user_age": user_age,
        "interests": interests, # interests has a many to many relationship 
    })

    print(f"Welcome {user_handle} !!!")
    return None

In [132]:
# testing create_user 
create_user()

Enter user handle:  Dan
Enter user age:  35
Enter two interests separated by a comma with no space (e.g. sports,python):  Python,Tennis


Welcome Dan !!!


In [133]:
# Checing users.json:
users.all()

[{'user_handle': 'Sam', 'user_age': 12, 'interests': ['Tennis', 'Python']},
 {'user_handle': 'Yis', 'user_age': 30, 'interests': ['Python', 'Reading']},
 {'user_handle': 'Dan', 'user_age': 35, 'interests': ['Python', 'Tennis']}]

Added users successfully 

# Multi-item query in Users.json for matching users:

In [134]:
# making a multi step query for my post creation stage: 

def match_users(user_handle,user_age): # checks if user_handle exists while having a matching age. 
    matched_user = users.search(
        (where("user_handle").matches(user_handle,flags=re.IGNORECASE)) &  # using matches() so I can use ignorecase
        (where("user_age") == user_age)
    )
    if not matched_user:
        print("User or age incorrect")
        return  

    return matched_user
    
match_users("sam",12)

[{'user_handle': 'Sam', 'user_age': 12, 'interests': ['Tennis', 'Python']}]

Create users function is working. 

This needed to query my DB for two features to ensure a correct match. 

# Create Posts and verify users: 

In [166]:
# Create post function: 

from datetime import datetime

def create_post():
    
    user_handle = input("Enter user handle: ")
    user_age = int(input("Enter user age: "))

    matched_users = match_users(user_handle, user_age)

    if not matched_users:
        return

    # Get the actual stored user from Users.json
    matched_user = matched_users[0]
    # if correct info was entered get post and update Posts.json
    post = input("Enter your post: ")
    time = datetime.now().isoformat() # iso format so it can be inserted into a json
    posts.insert({
        "user_handle":matched_user['user_handle'],
        "user_age":matched_user['user_age'],
        "post":post,
        "time":time})
    
    print(f"\n Posted your message:\n\n{post}")    
    return None


create_post()

Enter user handle:  Sam
Enter user age:  12
Enter your post:  sure



 Posted your message:

sure


### create_post function review: 

This step uses multiple database operations to verify the user and insert the post.

1. Query that the user exists with a matching age.
2. Insert the post and user information into `Posts.json` so posts can be tracked back to users later.  



# Check DB to see if it's working: 

In [167]:
posts.all()

[{'user_handle': 'Sam',
  'user_age': 12,
  'post': 'Hi',
  'time': '2026-09-30T19:19:25.209981'},
 {'user_handle': 'Yis',
  'user_age': 30,
  'post': 'Hi sam',
  'time': '2026-09-30T19:19:45.898394'},
 {'user_handle': 'Dan',
  'user_age': 35,
  'post': 'hi everyone',
  'time': '2026-09-30T19:19:55.604237'},
 {'user_handle': 'Dan',
  'user_age': 35,
  'post': 'anyone want to play tennis?',
  'time': '2026-09-30T19:20:11.943482'},
 {'user_handle': 'Sam',
  'user_age': 12,
  'post': 'sure',
  'time': '2026-09-30T19:20:19.308623'}]

We can see that posts are getting added each time in order

In [168]:
users.all()

[{'user_handle': 'Sam', 'user_age': 12, 'interests': ['Tennis', 'Python']},
 {'user_handle': 'Yis', 'user_age': 30, 'interests': ['Python', 'Reading']},
 {'user_handle': 'Dan', 'user_age': 35, 'interests': ['Python', 'Tennis']}]

# Aggregation and Reporting: 

In [169]:
# Count posts for each user

post_counts = {}

for post in posts.all():
    user_handle = post["user_handle"]

    if user_handle in post_counts:
        post_counts[user_handle] += 1
    else:
        post_counts[user_handle] = 1



average_posts = sum(post_counts.values()) / len(post_counts)


print("Post count for user:", post_counts)


print("Average posts by users:", average_posts)

Post count for user: {'Sam': 2, 'Yis': 1, 'Dan': 2}
Average posts by users: 1.6666666666666667


In [180]:
# 2. Find all posts from users who have Tennis as an interest (many-to-many relationship) 


# need to find tennis inside the list of interets
tennis_users = users.search(
    where("interests").any(
        lambda interest: interest.lower() == "tennis"
    )
)  

tennis_user_handles = [
    user["user_handle"]
    for user in tennis_users
]

tennis_posts = posts.search(
    where("user_handle").one_of(tennis_user_handles)
)


print(f"Total posts from Tennis users: {len(tennis_posts)}")


Total posts from Tennis users: 4


# Populating My Database to Test Scalability 

In [181]:
import random
import time
from datetime import datetime


def add_test_data():

    start_time = time.perf_counter()

    interests_list = [
        "Sports",
        "Python",
        "Travel",
        "Tennis"
    ]

    # Get existing handles once
    existing_handles = {
        user["user_handle"]
        for user in users.all()
    }

    # Build 1000 new users
    new_users = []
    user_number = 0

    while len(new_users) < 1000:

        user_handle = f"user{user_number}"

        if user_handle not in existing_handles:

            new_user = {
                "user_handle": user_handle,
                "user_age": random.randint(18, 70),
                "interests": random.sample(interests_list, 2)
            }

            new_users.append(new_user)
            existing_handles.add(user_handle)

        user_number += 1

    # Insert all 1000 users at once
    users.insert_multiple(new_users)

    # Build 100,000 posts
    new_posts = []

    for i in range(100000):

        user = random.choice(new_users)

        new_posts.append({
            "user_handle": user["user_handle"],
            "user_age": user["user_age"],
            "post": f"Test post {i}",
            "time": datetime.now().isoformat()
        })

    # Insert all 100,000 posts at once
    posts.insert_multiple(new_posts)

    elapsed_time = time.perf_counter() - start_time

    print("Added 1000 users and 100,000 posts.")
    print(f"Time taken: {elapsed_time:.2f} seconds")


add_test_data()

Added 1000 users and 100,000 posts.
Time taken: 0.24 seconds


Adding 1,000 users and 100,000 posts took less than one second. 



# Testing the DB: 

In [182]:
# Timing getting all posts
# Timing getting all users
import time
from tinydb import where

def get_all_users():
    start_time = time.perf_counter()
    
    users.all()

    end_time = time.perf_counter()

    elapsed_time = end_time - start_time
    return elapsed_time

    
def get_all_posts():
    start_time = time.perf_counter()
    
    posts.all()

    end_time = time.perf_counter()

    elapsed_time = end_time - start_time
    return elapsed_time

print(f'time to get all users {get_all_users()}')
print(f'time to get all posts {get_all_posts()}')

time to get all users 0.005545166997762863
time to get all posts 0.15240804200220737


Timing seems reasonable

In [183]:


# Time it takes to find all posts from one user

start = time.perf_counter()

user_posts = posts.search(
    where("user_handle") == "user5"
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(user_posts)}")

Time: 0.111174 seconds
Posts found: 108


In [184]:
# Time it takes to find all posts from users with age 30

start = time.perf_counter()

age_30_posts = posts.search(
    where("user_age") == 30
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(age_30_posts)}")

Time: 0.120059 seconds
Posts found: 1217


In [185]:
# Time it takes to find all posts from users between ages 20-40

start = time.perf_counter()

age_20_40_posts = posts.search(
    (where("user_age") >= 20) &
    (where("user_age") <= 40)
)

end = time.perf_counter()

print(f"Time: {end - start:.6f} seconds")
print(f"Posts found: {len(age_20_40_posts)}")

Time: 0.175248 seconds
Posts found: 36827


### All tests had sufficient speeds for my use case. The DB performed in a reasonable amount of time. 